# MedGemma v1.5 4B - Baseline, Ablation y LoRA

Notebook para la parte de Pablo:

1. Clonar el repo publico desde GitHub.
2. Instalar dependencias.
3. Hacer login en Hugging Face.
4. Probar MedGemma pretrained con texto e imagen.
5. Correr las 6 condiciones del pipeline mejorado.
6. Crear JSONL desde ACRIMA.
7. Arrancar un entrenamiento LoRA/QLoRA corto.

Antes de correr: cambia el runtime a GPU: `Runtime > Change runtime type > T4/L4 GPU`.

In [ ]:
# Configuracion principal
REPO_URL = "https://github.com/Luco1421/utils_medgemma.git"
REPO_DIR = "/content/utils_medgemma"

# Si copiaste el dataset ACRIMA a Drive, ajusta esta ruta despues de montar Drive.
# Debe contener carpetas: Glaucoma/ y Non Glaucoma/
DRIVE_DATASET_DIR = "/content/drive/MyDrive/MedGemma/dataset"

# Ruta local rapida dentro del runtime de Colab.
LOCAL_DATASET_DIR = f"{REPO_DIR}/dataset"

MODEL_ID = "google/medgemma-1.5-4b-it"

## 1. Clonar el repo publico

In [ ]:
import os, shutil

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

!git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!ls

## 2. Instalar dependencias

Si Colab pide reiniciar runtime, reinicia y vuelve a correr desde la celda de configuracion y `%cd {REPO_DIR}`.

In [ ]:
!pip install -q -r requirements-colab.txt

## 3. Login en Hugging Face

Primero acepta los terminos del modelo en Hugging Face:

https://huggingface.co/google/medgemma-1.5-4b-it

Luego pega tu token cuando se solicite.

In [ ]:
from huggingface_hub import login
login()

## 4. Montar Drive y copiar dataset

Si el dataset ya esta dentro del repo clonado, puedes saltarte la copia. Si esta en Drive, monta Drive y copia a `/content/utils_medgemma/dataset` para que sea mas rapido.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil

if os.path.exists(DRIVE_DATASET_DIR):
    if os.path.exists(LOCAL_DATASET_DIR):
        shutil.rmtree(LOCAL_DATASET_DIR)
    shutil.copytree(DRIVE_DATASET_DIR, LOCAL_DATASET_DIR)
    print(f"Dataset copiado a: {LOCAL_DATASET_DIR}")
else:
    print(f"No existe DRIVE_DATASET_DIR: {DRIVE_DATASET_DIR}")
    print("Si tu dataset ya esta en el repo, revisa LOCAL_DATASET_DIR.")

!find dataset -maxdepth 2 -type f | head -20

## 5. Inspeccionar dataset ACRIMA

Esperado:

```text
dataset/Glaucoma/*.jpg
dataset/Non Glaucoma/*.jpg
```

In [ ]:
!echo "Glaucoma:" && find dataset/Glaucoma -type f | wc -l
!echo "Non Glaucoma:" && find "dataset/Non Glaucoma" -type f | wc -l

import glob
glaucoma_images = sorted(glob.glob("dataset/Glaucoma/*"))
normal_images = sorted(glob.glob("dataset/Non Glaucoma/*"))
TEST_IMAGE = glaucoma_images[0] if glaucoma_images else normal_images[0]
print("TEST_IMAGE =", TEST_IMAGE)

## 6. Smoke test: solo texto

Esto prueba que MedGemma carga y genera texto sin imagen.

In [ ]:
!python experiments/smoke_medgemma_inference.py \
  --model-id {MODEL_ID} \
  --mode text \
  --prompt "Explain glaucoma in simple clinical terms." \
  --output results/text_only.json

## 7. Smoke test: imagen + texto

Esto prueba el baseline multimodal pretrained.

In [ ]:
!python experiments/smoke_medgemma_inference.py \
  --model-id {MODEL_ID} \
  --mode image-text \
  --image "$TEST_IMAGE" \
  --prompt "Describe the ophthalmological findings in this fundus image." \
  --output results/image_text_baseline.json

## 8. Ablation del pipeline mejorado

Corre A, B, C1, C2, D1, D2. Como ACRIMA no trae mascaras, usamos `--make-dummy-mask` solo para validar el flujo tecnico.

La dummy mask NO es resultado experimental; la mascara real debe venir de SAM/LoRA, WSSS o FSL/FD.

In [ ]:
!python experiments/run_ablation_baseline.py \
  --model-id {MODEL_ID} \
  --image "$TEST_IMAGE" \
  --make-dummy-mask \
  --prediction glaucoma \
  --distribution '{"glaucoma": 0.92, "normal": 0.08}' \
  --output results/ablation_dummy.json

## 9. Crear JSONL ACRIMA para prueba LoRA

Este JSONL se basa en etiquetas de carpeta, no en descripcion experta. Sirve para probar que el pipeline LoRA arranca.

In [ ]:
!python experiments/build_acrima_jsonl.py \
  --dataset-dir dataset \
  --output data/train_medgemma_acrima.jsonl \
  --max-per-class 20

!head -n 2 data/train_medgemma_acrima.jsonl

## 10. Entrenamiento LoRA/QLoRA corto

Empieza con `--max-steps 5`. El objetivo inicial es validar que el entrenamiento arranca, no obtener un modelo final.

In [ ]:
!python experiments/train_lora_medgemma.py \
  --model-id {MODEL_ID} \
  --train-jsonl data/train_medgemma_acrima.jsonl \
  --image-root dataset \
  --output-dir checkpoints/medgemma_lora_acrima_test \
  --use-qlora \
  --max-steps 5

## 11. Probar adapter LoRA

In [ ]:
!python experiments/smoke_medgemma_inference.py \
  --model-id {MODEL_ID} \
  --mode image-text \
  --image "$TEST_IMAGE" \
  --adapter-path checkpoints/medgemma_lora_acrima_test \
  --prompt "Describe the ophthalmological findings in this fundus image." \
  --output results/image_text_lora.json

## 12. Guardar resultados en Drive

In [ ]:
!mkdir -p /content/drive/MyDrive/MedGemma/outputs
!cp -r results /content/drive/MyDrive/MedGemma/outputs/results_colab
!cp -r checkpoints /content/drive/MyDrive/MedGemma/outputs/checkpoints_colab || true
!find /content/drive/MyDrive/MedGemma/outputs -maxdepth 3 -type f | head -30

## Que decir en la reunion

- El dataset actual es ACRIMA por carpetas de clase: glaucoma/no glaucoma.
- Sirve para validar inferencia baseline y LoRA tecnico.
- No trae mascaras ni descripciones expertas, por eso no reemplaza el dataset final del paper.
- La ablation A/B/C1/C2/D1/D2 ya esta implementada.
- La mascara dummy solo valida flujo; la mascara real debe venir del equipo SAM/WSSS/FSL-FD.
- LoRA queda preparado para entrenar adapters cuando haya ejemplos imagen-prompt-respuesta curados.